<a href="https://colab.research.google.com/github/maddi-venkata-teja/Algoverse-Chatbot/blob/main/A24126552142_tanusree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required libraries
!pip install -q langchain-community==0.2.16 langchain-core==0.2.38 jq==1.8.0
print("✅ Done!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.4/396.4 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 748.9/748.9 kB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 

In [ ]:
import os
import json
import re
import IPython
from langchain_community.document_loaders import TextLoader, JSONLoader, DirectoryLoader
from langchain_core.documents import Document

# =====================================================================
# DOCUMENT LOADERS — LangChain Framework
# =====================================================================

# ------------------------------------------------------------------
# Loader 1: TextLoader
# ------------------------------------------------------------------
print("=" * 60)
print("LOADER 1: TextLoader")
print("=" * 60)
text_loader = TextLoader("dsa.json")
text_docs = text_loader.load()
print(f"Source file : dsa.json")
print(f"Documents   : {len(text_docs)}")
print(f"Metadata    : {text_docs[0].metadata}")
print("Content (first 300 chars):")
print(text_docs[0].page_content[:300])

# ------------------------------------------------------------------
# Loader 2: JSONLoader
# ------------------------------------------------------------------
print()
print("=" * 60)
print("LOADER 2: JSONLoader")
print("=" * 60)
json_loader = JSONLoader(
    file_path="dsa.json",
    jq_schema=".dsa_concepts[]",
    text_content=False
)
json_docs = json_loader.load()
print(f"Source file : dsa.json")
print(f"Documents   : {len(json_docs)} (one per DSA concept)")
print(f"Metadata    : {json_docs[0].metadata}")
print("Content of concept 0 (first 400 chars):")
print(json_docs[0].page_content[:400])

# ------------------------------------------------------------------
# Loader 3: DirectoryLoader
# ------------------------------------------------------------------
print()
print("=" * 60)
print("LOADER 3: DirectoryLoader")
print("=" * 60)
os.makedirs("./sample_data", exist_ok=True)
dir_loader = DirectoryLoader("./sample_data/", glob="**/*.json", loader_cls=TextLoader)
dir_docs = dir_loader.load()
print(f"Source folder : ./sample_data/")
print(f"Documents     : {len(dir_docs)} (one per .json file found)")
for d in dir_docs:
    print(f"  - {d.metadata.get('source', 'unknown')}")

print()
print("=" * 60)
print("ALL LOADERS OUTPUT THE SAME FORMAT:")
print("  doc.page_content -> plain text string (readable by model)")
print("  doc.metadata     -> dict with source, page, etc.")
print("=" * 60)
print(f"  TextLoader      -> {len(text_docs)} document(s)")
print(f"  JSONLoader      -> {len(json_docs)} document(s)")
print(f"  DirectoryLoader -> {len(dir_docs)} document(s)")
print()
print("✅ All loaders working!")

LOADER 1: TextLoader
Source file : dsa.json
Documents   : 1
Metadata    : {'source': 'dsa.json'}
Content (first 300 chars):
{
  "dsa_concepts": [
    {
      "name": "Array",
      "category": "Linear Data Structure",
      "description": "An array is a collection of elements stored at contiguous memory locations. It is the simplest and most widely used data structure. Each element is identified by an index starting from

LOADER 2: JSONLoader
Source file : dsa.json
Documents   : 15 (one per DSA concept)
Metadata    : {'source': '/content/dsa.json', 'seq_num': 1}
Content of concept 0 (first 400 chars):
{"name": "Array", "category": "Linear Data Structure", "description": "An array is a collection of elements stored at contiguous memory locations. It is the simplest and most widely used data structure. Each element is identified by an index starting from 0.", "operations": ["access", "search", "insert", "delete", "traverse"], "time_complexity": {"access": "O(1)", "search": "O(n)", "inser

In [ ]:
# =====================================================================
# DSA CHATBOT — using ipywidgets (works reliably in Colab)
# =====================================================================
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Load dsa.json
with open("dsa.json", "r") as f:
    raw_data = json.load(f)

# Build lookup
dsa_lookup = {}
for concept in raw_data["dsa_concepts"]:
    name = concept.get("name", "").lower()
    dsa_lookup[name] = concept

def get_answer(question):
    question_lower = question.lower()
    for key, concept in dsa_lookup.items():
        if key in question_lower:
            name      = concept.get("name", "")
            desc      = concept.get("description", "")
            tc        = concept.get("time_complexity", {})
            use_cases = concept.get("use_cases", [])
            adv       = concept.get("advantages", [])
            disadv    = concept.get("disadvantages", [])
            answer = (
                f"<b>{name}</b><br>"
                f"{'='*40}<br>"
                f"{desc}<br><br>"
                f"<b>Time Complexity:</b> {json.dumps(tc)}<br><br>"
                f"<b>Advantages:</b><br>" + "<br>".join(f"• {a}" for a in adv) +
                f"<br><br><b>Disadvantages:</b><br>" + "<br>".join(f"• {d}" for d in disadv) +
                f"<br><br><b>Use Cases:</b><br>" + "<br>".join(f"• {u}" for u in use_cases)
            )
            return answer
    topics = ", ".join([c.get("name","") for c in raw_data["dsa_concepts"]])
    return f"Topic not found. Available topics: {topics}"

# ── UI Components ──────────────────────────────────────────────────
title = HTML("<h2 style='color:#4CAF50'>🤖 DSA Expert Chatbot</h2><p style='color:#aaa'>Ask about: Array, Stack, Queue, Graph, Tree, Heap, Trie, Sorting, Searching, Recursion, Dynamic Programming</p>")

chat_output = widgets.Output(layout=widgets.Layout(
    border="2px solid #4CAF50",
    min_height="300px",
    max_height="400px",
    overflow_y="auto",
    padding="10px",
    background_color="#1e1e1e"
))

text_input = widgets.Text(
    placeholder="e.g. explain stack",
    layout=widgets.Layout(width="500px", height="40px")
)

send_btn = widgets.Button(
    description="Send",
    button_style="success",
    layout=widgets.Layout(width="100px", height="40px")
)

# Show welcome message
with chat_output:
    display(HTML("<div style='color:white;background:#1565c0;padding:10px;border-radius:8px;margin:5px 0'>👋 Hello! Ask me anything about Data Structures and Algorithms!</div>"))

def on_send(b):
    question = text_input.value.strip()
    if not question:
        return
    text_input.value = ""
    answer = get_answer(question)
    with chat_output:
        display(HTML(f"<div style='color:white;background:#2e7d32;padding:10px;border-radius:8px;margin:5px 0;text-align:right'>You: {question}</div>"))
        display(HTML(f"<div style='color:white;background:#1565c0;padding:10px;border-radius:8px;margin:5px 0'>🤖 {answer}</div>"))

def on_enter(change):
    if change["new"].endswith("\n") or text_input.value:
        pass

send_btn.on_click(on_send)

# Also send on Enter key
def handle_submit(widget):
    on_send(None)

text_input.on_submit(handle_submit)

# Display chatbot
display(title)
display(chat_output)
display(widgets.HBox([text_input, send_btn]))
print("✅ DSA Chatbot is ready! Type a topic and press Send or Enter.")

Output(layout=Layout(border='2px solid #4CAF50', max_height='400px', min_height='300px', overflow_y='auto', pa…

✅ DSA Chatbot is ready! Type a topic and press Send or Enter.
